# Multi-Factor Strategy — Backtest

**Goal:** Compare the multi-factor strategy vs. the best single-feature baseline.

**IS/OOS split:**
- In-Sample (IS): 2016-01-01 → 2021-12-31
- Out-of-Sample (OOS): 2022-01-01 → 2025-12-31

**Baseline:** Best single-feature strategy from prior research — MA crossover (inverted) on HP trend, IS Sharpe ≈ 3.7.

**Multi-factor features used:**
1. `ma_spread_on_trend` (inverted, IC −0.318)
2. `trend_deviation_from_ma` (inverted, IC −0.344)
3. `ma_crossover_signal` (inverted, IC −0.397)
4. `trend_strength` (regime feature)

**Key question:** Does combining features improve Sharpe or reduce drawdown vs. best single feature?

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import sqrt

from src.data.loader import FXDataLoader
from src.features import (
    compute_log_returns, atr, causal_hp_trend,
    ma_crossover_signal,
)
from src.features.generators import crossing_threshold_signal
from src.backtest.engine import VectorizedBacktest
from src.backtest.cost_model import FXCostModel
from src.strategies.multi_factor import run_multi_factor_backtest, MultiFactorConfig

plt.style.use('seaborn-v0_8-darkgrid')
print('Imports OK')

## 1. Load data and define IS/OOS splits

In [ ]:
loader = FXDataLoader('../data/raw')
df = loader.load('USDJPY_10yr_1h_dukascopy')

IS_END   = '2021-12-31'
OOS_START = '2022-01-01'

df_is  = df[df.index <= IS_END]
df_oos = df[df.index >= OOS_START]

print(f'IS  ({IS_END[:4]} end):  {len(df_is):,} bars   {df_is.index[0].date()} → {df_is.index[-1].date()}')
print(f'OOS ({OOS_START[:4]} start): {len(df_oos):,} bars   {df_oos.index[0].date()} → {df_oos.index[-1].date()}')

cost_model = FXCostModel(spread_bps=0.7, slippage_bps=0.2)
BARS_PER_YEAR = 252 * 24

def sharpe(net_returns):
    m = net_returns.mean()
    s = net_returns.std()
    return (m / s) * sqrt(BARS_PER_YEAR) if s > 0 else np.nan

def max_dd(equity):
    return float(((equity - equity.cummax()) / equity.cummax()).min())

def annret(net_returns):
    return float((1 + net_returns.mean()) ** BARS_PER_YEAR - 1)

## 2. Baseline: best single-feature (inverted MA crossover)

In [ ]:
close = df['close']
high = df['high']
low = df['low']
log_ret = compute_log_returns(close)
atr_s = atr(high, low, close, window=20)

print('Building baseline signal (HP trend + MA crossover)...')
trend = causal_hp_trend(close, lamb=3.9e9, window=500)
crossover = ma_crossover_signal(trend, t1=72, t2=240)
# Inverted: crossover UP → SHORT, crossover DOWN → LONG
baseline_signal = -crossover.fillna(0.0)
print('Done.')

bt_is_base  = VectorizedBacktest(df_is,  baseline_signal[df_is.index],  cost_model)
bt_oos_base = VectorizedBacktest(df_oos, baseline_signal[df_oos.index], cost_model)
res_is_base  = bt_is_base.run()
res_oos_base = bt_oos_base.run()

print(f'Baseline IS   Sharpe={sharpe(res_is_base["net_returns"]):.2f}  AnnRet={annret(res_is_base["net_returns"]):.1%}  MaxDD={max_dd(res_is_base["equity"]):.1%}')
print(f'Baseline OOS  Sharpe={sharpe(res_oos_base["net_returns"]):.2f}  AnnRet={annret(res_oos_base["net_returns"]):.1%}  MaxDD={max_dd(res_oos_base["equity"]):.1%}')

## 3. Multi-factor IS and OOS backtest

In [ ]:
config = MultiFactorConfig(
    signal_mode='binary_threshold',
    threshold=0.3,
    norm_method='zscore',
    spread_bps=0.7,
    slippage_bps=0.2,
)

print('Running multi-factor IS backtest...')
res_is_mf  = run_multi_factor_backtest(df_is, config)
print(f'IS   Sharpe={res_is_mf["sharpe"]:.2f}  AnnRet={res_is_mf["annual_return"]:.1%}  MaxDD={res_is_mf["max_drawdown"]:.1%}  n_trades={res_is_mf["n_trades"]}')

print('Running multi-factor OOS backtest...')
res_oos_mf = run_multi_factor_backtest(df_oos, config)
print(f'OOS  Sharpe={res_oos_mf["sharpe"]:.2f}  AnnRet={res_oos_mf["annual_return"]:.1%}  MaxDD={res_oos_mf["max_drawdown"]:.1%}  n_trades={res_oos_mf["n_trades"]}')

## 4. Results comparison table

In [ ]:
comparison = pd.DataFrame([
    {
        'Strategy': 'Baseline (inverted crossover)',
        'Period': 'IS',
        'Sharpe': round(sharpe(res_is_base['net_returns']), 2),
        'Ann. Return': f"{annret(res_is_base['net_returns']):.1%}",
        'Max Drawdown': f"{max_dd(res_is_base['equity']):.1%}",
        'N Trades': bt_is_base.n_trades,
    },
    {
        'Strategy': 'Baseline (inverted crossover)',
        'Period': 'OOS',
        'Sharpe': round(sharpe(res_oos_base['net_returns']), 2),
        'Ann. Return': f"{annret(res_oos_base['net_returns']):.1%}",
        'Max Drawdown': f"{max_dd(res_oos_base['equity']):.1%}",
        'N Trades': bt_oos_base.n_trades,
    },
    {
        'Strategy': 'Multi-Factor (4 features)',
        'Period': 'IS',
        'Sharpe': round(res_is_mf['sharpe'], 2),
        'Ann. Return': f"{res_is_mf['annual_return']:.1%}",
        'Max Drawdown': f"{res_is_mf['max_drawdown']:.1%}",
        'N Trades': res_is_mf['n_trades'],
    },
    {
        'Strategy': 'Multi-Factor (4 features)',
        'Period': 'OOS',
        'Sharpe': round(res_oos_mf['sharpe'], 2),
        'Ann. Return': f"{res_oos_mf['annual_return']:.1%}",
        'Max Drawdown': f"{res_oos_mf['max_drawdown']:.1%}",
        'N Trades': res_oos_mf['n_trades'],
    },
])

comparison.set_index(['Strategy', 'Period'])

## 5. Equity curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

plot_configs = [
    (axes[0, 0], res_is_base,  'Baseline IS',          'royalblue'),
    (axes[0, 1], res_is_mf,    'Multi-Factor IS',       'darkorange'),
    (axes[1, 0], res_oos_base, 'Baseline OOS',          'royalblue'),
    (axes[1, 1], res_oos_mf,   'Multi-Factor OOS',      'darkorange'),
]

for ax, res, title, color in plot_configs:
    equity = res['equity']
    nr = res['net_returns']
    sh = res.get('sharpe', sharpe(nr))
    dd = max_dd(equity)

    ax.plot(equity.index, equity.values, lw=1.2, color=color)
    ax.fill_between(
        equity.index,
        equity.values,
        equity.cummax().values,
        alpha=0.2, color='red', label='Drawdown',
    )
    ax.axhline(1.0, color='black', lw=0.5, linestyle='--')
    ax.set_title(
        f'{title}\nSharpe={sh:.2f}  MaxDD={dd:.1%}',
        fontsize=11,
    )
    ax.set_ylabel('Equity (starting at 1.0)')
    ax.legend(fontsize=9)

plt.suptitle('Multi-Factor vs. Baseline — Equity Curves', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../reports/multi_factor_equity_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: reports/multi_factor_equity_curves.png')

## 6. Interpretation and next steps

In [ ]:
is_sharpe_mf   = res_is_mf['sharpe']
is_sharpe_base = sharpe(res_is_base['net_returns'])
oos_sharpe_mf  = res_oos_mf['sharpe']
oos_sharpe_base = sharpe(res_oos_base['net_returns'])

print('=== INTERPRETATION ===')
if oos_sharpe_mf > oos_sharpe_base:
    print(f'✅ Multi-factor IMPROVES OOS Sharpe: {oos_sharpe_base:.2f} → {oos_sharpe_mf:.2f} (+{oos_sharpe_mf - oos_sharpe_base:.2f})')
else:
    print(f'⚠️  Multi-factor does NOT improve OOS Sharpe: {oos_sharpe_base:.2f} → {oos_sharpe_mf:.2f}')
    print('   Consider: tuning threshold, removing trend_strength feature, or using ensemble_vote mode.')

is_dd_mf   = res_is_mf['max_drawdown']
is_dd_base = max_dd(res_is_base['equity'])
if abs(is_dd_mf) < abs(is_dd_base):
    print(f'✅ Multi-factor REDUCES IS max drawdown: {is_dd_base:.1%} → {is_dd_mf:.1%}')

print()
print('Next steps:')
print('  1. Run feature selection (selection.py) to confirm which features pass all 5 criteria.')
print('  2. Try ensemble_vote mode — less sensitive to feature scale differences.')
print('  3. Add vol_regime filter: restrict to Medium-Vol periods only (peak IC regime).')
print('  4. Grid search threshold ∈ [0.1, 0.5] for binary_threshold mode.')